In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="classla/ParlaSpeech-RS", 
    repo_type="dataset", local_dir="./ParlaSpeech-RS", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 138 files: 100%|██████████| 138/138 [00:54<00:00,  2.53it/s]


'/home/ubuntu/ParlaSpeech-RS'

In [3]:
files = glob('ParlaSpeech-RS/*/*.parquet')
len(files)

138

In [5]:
df = pd.read_parquet(files[0])
df.head()

,id,audio,text,text_cyrillic,text_normalised,text_cyrillic_normalised,words,audio_length,date,speaker_name,speaker_gender,speaker_birth,speaker_party,party_orientation,party_status
0,ParlaMint-RS_2022-02-02-0.u12131_16048-16097,"{'bytes': b'fLaC\x00\x00\x00""\x04\x80\x04\x80\...",Srbija je finansijski podržala građane i privr...,Србија је финансијски подржала грађане и привр...,Srbija je finansijski podržala građane i privr...,Србија је финансијски подржала грађане и привр...,"[{'char_e': 6, 'char_s': 0, 'time_e': 0.28, 't...",2.86,2022-02-02,"Bakarec, Nebojša",M,1963,SNS,Big tent,Coalition
1,ParlaMint-RS_2022-02-02-0.u12131_16099-16258,"{'bytes': b'fLaC\x00\x00\x00""\x04\x80\x04\x80\...",Setimo se 100 evra pomoći punoletnim građanima...,Сетимо се 100 евра помоћи пунолетним грађанима...,Setimo se 100 evra pomoći punoletnim građanima...,Сетимо се 100 евра помоћи пунолетним грађанима...,"[{'char_e': 6, 'char_s': 0, 'time_e': 0.38, 't...",11.78,2022-02-02,"Bakarec, Nebojša",M,1963,SNS,Big tent,Coalition
2,ParlaMint-RS_2022-02-02-0.u12131_16260-16352,"{'bytes': b'fLaC\x00\x00\x00""\x04\x80\x04\x80\...",Druga tura pomoći nezaposlenima je bila opet 6...,Друга тура помоћи незапосленима је била опет 6...,Druga tura pomoći nezaposlenima je bila opet 6...,Друга тура помоћи незапосленима је била опет 6...,"[{'char_e': 5, 'char_s': 0, 'time_e': 0.28, 't...",6.96,2022-02-02,"Bakarec, Nebojša",M,1963,SNS,Big tent,Coalition
3,ParlaMint-RS_2022-02-02-0.u12131_16441-16542,"{'bytes': b'fLaC\x00\x00\x00""\x04\x80\x04\x80\...","Ono što je jako važno, prvi put su predete, i ...","Оно што је јако важно, први пут су предете, и ...","Ono što je jako važno, prvi put su predete, i ...","Оно што је јако важно, први пут су предете, и ...","[{'char_e': 3, 'char_s': 0, 'time_e': 0.08, 't...",8.92,2022-02-02,"Bakarec, Nebojša",M,1963,SNS,Big tent,Coalition
4,ParlaMint-RS_2022-02-02-0.u12131_16544-16566,"{'bytes': b'fLaC\x00\x00\x00""\x04\x80\x04\x80\...",Pomognute su porodilje.,Помогнуте су породиље.,Pomognute su porodilje.,Помогнуте су породиље.,"[{'char_e': 9, 'char_s': 0, 'time_e': 0.58, 't...",1.92,2022-02-02,"Bakarec, Nebojša",M,1963,SNS,Big tent,Coalition


In [10]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['speaker_name'].iloc[i]}"
            })

            t = df['text_cyrillic'].iloc[i].strip()
            if len(t) < 2:
                continue

            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['speaker_name'].iloc[i]}"
            })
        
    return data

In [8]:
data = loop((files[:1], 0))

100%|██████████| 1/1 [00:00<00:00,  5.40it/s]


In [11]:
data = multiprocessing(files, loop, cores = 40)

100%|██████████| 3/3 [06:39<00:00, 133.23s/it]


In [12]:
len(data)

551666

In [13]:
data[:2]

[{'audio_filename': 'ParlaSpeech-RS_audio/ParlaSpeech-RS-data-train-00137-of-00138_0.mp3',
  'text': 'Srbija je finansijski podržala građane i privredu.',
  'speaker': 'ParlaSpeech-RS_audio_Bakarec, Nebojša'},
 {'audio_filename': 'ParlaSpeech-RS_audio/ParlaSpeech-RS-data-train-00137-of-00138_0.mp3',
  'text': 'Србија је финансијски подржала грађане и привреду.',
  'speaker': 'ParlaSpeech-RS_audio_Bakarec, Nebojša'}]

In [14]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'ParlaSpeech-RS_audio/ParlaSpeech-RS-data-train-00137-of-00138_0.mp3',
 'text': 'Srbija je finansijski podržala građane i privredu.',
 'speaker': 'ParlaSpeech-RS_audio_Bakarec, Nebojša'}

In [15]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'ParlaSpeech-RS')

Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00,  5.99ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  99%|█████████▉| 60.8MB / 61.3MB,  152MB/s  
Processing Files (1 / 1): 100%|██████████| 61.3MB / 61.3MB, 51.1MB/s  
Processing Files (1 / 1): 100%|██████████| 61.3MB / 61.3MB, 17.0MB/s  
New Data Upload: 100%|██████████| 61.3MB / 61.3MB, 17.0MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:04<00:00,  4.77s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/8909aac8aabfcb8873bb81767643f7d637801e64', commit_message='Upload dataset', commit_description='', oid='8909aac8aabfcb8873bb81767643f7d637801e64', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [16]:
audio_files = [d['audio_filename'] for d in data]

with open('ParlaSpeech-RS-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [19]:
# !zip -rq ParlaSpeech-RS_audio.zip ParlaSpeech-RS_audio

In [20]:
# !hf upload malaysia-ai/Multilingual-TTS ParlaSpeech-RS_audio.zip --repo-type=dataset